![Databricks Academy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/db-academy.png)

# 02 - REQUIRED - Course Setup and Creating a Pipeline

## Overview

In this demo, we'll set up the course environment, explore its components, build a traditional ETL pipeline using JSON files as the data source, and then learn how to create a sample Apache Spark™ Declarative Pipeline (SDP).

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Navigate the Workspace** to locate course catalogs, schemas, and source files.
2. **Create an Apache Spark™ Declarative Pipeline** using the Workspace and the Pipeline UI.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the cell below to initialize your environment.

    This setup step does the following:

    - **Assumes you have permission to create a catalog** when running outside of a Databricks-provided Vocareum workspace.
    - Creates three schemas in your specified catalog:
        - **sdp_1_bronze**
        - **sdp_2_silver**
        - **sdp_3_gold**
    - Creates three volumes in your **YOUR_LABUSER_CATALOG.sdp_1_bronze.source** volume and adds a single JSON file in each volume.
    - Checks your specified Serverless compute version.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

## B. Explore the Lab Environment

Explore the raw data source files, catalogs, and schemas in the course lab environment.



### B1. View Your Catalog and Schemas

1. Complete these steps to explore your user catalog and schemas:

   a. Select the **Catalog** icon ![Catalog Icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/catalog_icon.png) in the left navigation bar.

   b. You should see your unique catalog, named something like **labuser_USERNAME**. You will use this catalog throughout the course.

   c. Expand your **labuser** catalog. It should contain the following schemas:
      - **sdp_1_bronze**
      - **sdp_2_silver**
      - **sdp_3_gold**
      - **default**

### B2. View the Streaming Source Files

1. Complete the following steps to view where your streaming raw source files are located:

   a. Select the **Catalog** icon ![Catalog Icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/catalog_icon.png) in the left navigation bar.

   b. Expand your **labuser_USERNAME** catalog.

   c. Expand the **sdp_1_bronze** schema

   d. Expand your **source** volume. You should see three folders:
      - **customers**
      - **orders**
      - **status**

   e. Expand each folder and notice that each location contains a single JSON file to start with.

### B3. Confirm the Source Volume Path

To easily reference the volume path (`/Volumes/LABUSER_YOURUSERNAME/sdp_1_bronze/source`) throughout the course, use the following variables:

- **Python:** `source_volume_path`
- **SQL:** `source_volume_path`

Run the cells below and confirm the path points to your volume.

**Example:** `/Volumes/LABUSER_YOURUSERNAME/sdp_1_bronze/source`

In [0]:
## With Python
print(source_volume_path)

In [0]:
%sql
-- With SQL
values(source_volume_path)

## C. Build a Traditional ETL Pipeline


### C1. Preview the Raw Orders Data

1. Query the raw JSON file(s) in your `/Volumes/labuser/sdp_1_bronze/source/orders` volume to preview the data.

   The JSON file is displayed in tabular form using the `read_files` function. 

   Notice the following:

   a. The **orders** JSON file contains order data for a company.

   b. The one JSON file in your **source/orders** volume (**00.json**) contains 174 rows. Remember that number for later.

In [0]:
spark.sql(f'''
          SELECT * 
          FROM json.`{source_volume_path}/orders`
          ''').display()

### C2. Write a Traditional ETL Pipeline

1. Traditionally, you would build an ETL pipeline by reading all files within a cloud storage location each time the pipeline runs. As data scales, this method becomes inefficient, more expensive, and time-consuming.

   **NOTES:**
   - The tables and views will be written to your **labuser.sdp_1_bronze** schema (database).
   - Knowledge of the Databricks `read_files` function is a prerequisite for this course.
      - `read_files` table-valued function: [AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files) |[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/read_files) |
      [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/read_files)

In [0]:
%sql
--------------------------------------------
-- JSON -> Bronze
--------------------------------------------
-- Read ALL files from your working directory each time the query is executed
CREATE OR REPLACE TABLE sdp_1_bronze.orders_bronze
AS
SELECT
  *,
  current_timestamp() AS processing_time,
  _metadata.file_name AS source_file
FROM read_files(
    source_volume_path || "/orders",
    format =>"json");

--------------------------------------------
-- Bronze -> Silver
--------------------------------------------
-- Read the entire bronze table each time the query is executed
CREATE OR REPLACE TABLE sdp_1_bronze.orders_silver
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp,
  customer_id,
  notifications
FROM sdp_1_bronze.orders_bronze;

--------------------------------------------
-- Silver -> Gold View
--------------------------------------------
-- Aggregate the silver each time the query is executed.
CREATE OR REPLACE VIEW sdp_1_bronze.orders_by_date_vw
AS
SELECT
  date(order_timestamp) AS order_date,
  count(*) AS total_daily_orders
FROM sdp_1_bronze.orders_silver
GROUP BY date(order_timestamp);

### C3. Preview the Pipeline Results

1. Run the cells below to preview the:
    - **orders_bronze** table
    - **orders_silver** table
    - **orders_by_date_vw** view. 

    Explore the results.

In [0]:
%sql
SELECT *
FROM sdp_1_bronze.orders_bronze
LIMIT 5;

In [0]:
%sql
SELECT *
FROM sdp_1_bronze.orders_silver
LIMIT 5;

In [0]:
%sql
SELECT *
FROM sdp_1_bronze.orders_by_date_vw
LIMIT 5;

### C4. Traditional Batch Pipeline Limitations

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
<style>
.info-wrapper {
  background: #F9F7F4;
  border-radius: 10px;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  padding: 32px;
  position: relative;
}
.info-wrapper::before {
  content: "";
  position: absolute;
  top: 0;
  left: 0;
  width: 100%;
  height: 6px;
  background: #FF5F46;
  border-radius: 10px 10px 0 0;
}
.info-header {
  font-size: 20pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 8px;
}
.info-subtitle {
  font-size: 14pt;
  color: #5A6F77;
  margin-bottom: 28px;
}
.info-row {
  display: flex;
  align-items: flex-start;
  gap: 20px;
  padding: 20px 0;
  border-bottom: 1px solid #EEEDE9;
}
.info-row:last-child {
  border-bottom: none;
}
.info-accent {
  font-size: 24pt;
  font-weight: 800;
  line-height: 1;
  flex-shrink: 0;
  width: 44px;
  text-align: center;
}
.info-content {
  flex: 1;
}
.info-title {
  font-size: 16pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 4px;
}
.info-desc {
  font-size: 14pt;
  color: #5A6F77;
  line-height: 1.5;
}
.info-tag {
  font-size: 14pt;
  color: #618794;
  font-weight: 700;
  flex-shrink: 0;
  white-space: nowrap;
  padding: 6px 10px;
  background: #EEF4F7;
  border-radius: 999px;
  margin-top: 2px;
}
</style>
<div class="info-wrapper">
  <div class="info-header">Traditional Batch Pipeline Limitations</div>
  <div class="info-subtitle">Why this current ETL approach struggles as data grows</div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">Bronze: Full Re-Read Every Run</div>
      <div class="info-desc">Every execution reads <strong>all files</strong> from cloud storage, not just new ones. Cost and runtime grow with every file added.</div>
    </div>
    <div class="info-tag">Efficiency</div>
  </div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">Silver: Always Reprocesses All Rows</div>
      <div class="info-desc">The silver layer reads the entire bronze table on every run. No incremental processing. A full scan every time.</div>
    </div>
    <div class="info-tag">Efficiency</div>
  </div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">Views Re-Execute on Every Call</div>
      <div class="info-desc"><strong><code>orders_by_date_vw</code></strong> runs the underlying query each time it is referenced. No caching, no optimization as data scales.</div>
    </div>
    <div class="info-tag">Performance</div>
  </div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">Data Quality Requires Extra Code</div>
      <div class="info-desc">Checking for invalid or missing values means writing and maintaining additional validation logic outside the pipeline.</div>
    </div>
    <div class="info-tag">Quality</div>
  </div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">Run Monitoring Is a Challenge</div>
      <div class="info-desc">Tracking success, failure, and row counts across runs requires custom logging. There is no built-in visibility.</div>
    </div>
    <div class="info-tag">Observability</div>
  </div>
  <div class="info-row">
    <div class="info-accent" style="color: #FF5F46;">&#9679;</div>
    <div class="info-content">
      <div class="info-title">No UI to Explore or Fix Issues</div>
      <div class="info-desc">When something goes wrong, there is no simple interface to inspect, debug, or remediate problems from run to run.</div>
    </div>
    <div class="info-tag">Observability</div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>
<summary>Click to expand</summary>

<h3>Bronze: Full Re-Read Every Run</h3>
<ul>
<li>The batch code uses <code>spark.read</code>, a static, full read. It does not use <code>readStream</code> (streaming/incremental).</li>
<li>Every execution scans every file in the volume, even files already processed in prior runs.</li>
<li>As months of data accumulate, the same old files get re-read on every trigger.</li>
<li>Cost scales with total file count, not with the volume of new data.</li>
</ul>

<h3>Silver: Always Reprocesses All Rows</h3>
<ul>
<li>The silver transformation reads all rows from bronze, not just rows added since the last run.</li>
<li>Even a simple filter or join runs against the full dataset every time.</li>
<li>This compounds with bronze ingestion — you are already paying to re-read files, then re-transforming all of the results.</li>
</ul>

<h3>Views Re-Execute on Every Call</h3>
<ul>
<li>Traditional views are just saved SQL. They do not actually store results.</li>
<li><code>orders_by_date_vw</code> queries the silver table fresh each time it is referenced.</li>
<li>In a dashboard or downstream query context, this means redundant full scans with no caching benefit.</li>
</ul>

<h3>Data Quality Requires Extra Code</h3>
<ul>
<li>Nothing in the traditional pipeline enforces constraints on incoming data.</li>
<li>You would need to write separate validation logic: <code>WHERE</code> clauses, <code>CASE</code> statements, or custom assertions.</li>
<li>This logic lives outside the pipeline and is easy to forget, skip, or break during refactoring.</li>
<li>There is no standard pattern for what to do when bad data arrives — drop it, halt the run, or log it somewhere?</li>
</ul>

<h3>Run Monitoring Is a Challenge</h3>
<ul>
<li>Batch jobs expose success/failure at the job level, but not at the data level.</li>
<li>How many rows were written? Were there duplicates? Did any files fail silently?</li>
<li>You often will not know there is a problem until a downstream user reports something wrong.</li>
</ul>

<h3>No UI to Explore or Fix Issues</h3>
<ul>
<li>When a run fails or produces bad data, debugging means going back to code and logs.</li>
<li>There is no lineage view, no data preview per stage, and no way to see what changed between runs.</li>
<li>Fixing one issue often requires re-running the entire pipeline from scratch.</li>
</ul>

<p><strong>Transition to SDP:</strong> Each of these problems has a direct solution in Spark Declarative Pipelines — incremental processing, built-in data quality constraints, a pipeline UI with lineage, and event-driven monitoring.</p>

</details>

### C5. Traditional Structured Streaming Pipeline Limitations

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
<style>
.ss-wrapper {
  background: #F9F7F4;
  border-radius: 10px;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  padding: 32px;
  position: relative;
  margin-top: 24px;
}
.ss-wrapper::before {
  content: "";
  position: absolute;
  top: 0;
  left: 0;
  width: 100%;
  height: 6px;
  background: #FF3621;
  border-radius: 10px 10px 0 0;
}
.ss-header {
  font-size: 20pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 8px;
}
.ss-subtitle {
  font-size: 14pt;
  color: #5A6F77;
  margin-bottom: 24px;
}
.ss-body {
  font-size: 14pt;
  color: #0b2026;
  line-height: 1.6;
  margin-bottom: 20px;
}
.ss-row {
  display: flex;
  align-items: flex-start;
  gap: 20px;
  padding: 20px 0;
  border-bottom: 1px solid #EEEDE9;
}
.ss-row:last-child {
  border-bottom: none;
}
.ss-accent {
  font-size: 24pt;
  font-weight: 800;
  line-height: 1;
  flex-shrink: 0;
  width: 44px;
  text-align: center;
  color: #FFAB00;
}
.ss-content {
  flex: 1;
}
.ss-title {
  font-size: 16pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 4px;
}
.ss-desc {
  font-size: 14pt;
  color: #5A6F77;
  line-height: 1.5;
}
.ss-tag {
  font-size: 14pt;
  color: #618794;
  font-weight: 700;
  flex-shrink: 0;
  white-space: nowrap;
  padding: 6px 10px;
  background: #EEF4F7;
  border-radius: 999px;
  margin-top: 2px;
}
.ss-takeaway {
  background: #EEEDE9;
  border-radius: 8px;
  padding: 16px 20px;
  margin-top: 24px;
}
.ss-takeaway-text {
  font-size: 14pt;
  color: #0b2026;
  line-height: 1.6;
}
</style>

<div class="ss-wrapper">
  <div class="ss-header">"But Can't We Just Use Structured Streaming?"</div>
  <div class="ss-subtitle">Incremental processing is possible, but it comes at a cost of complexity</div>

  <div class="ss-body">
    You <em>could</em> replace <code style="background: #EEEDE9; padding: 2px 6px; border-radius: 4px;">the pipeline</code> with <code style="background: #EEEDE9; padding: 2px 6px; border-radius: 4px;">spark.readStream</code> to get incremental ingestion. But that means rewriting your pipeline with significantly more code and operational overhead.
  </div>

  <div class="ss-row">
    <div class="ss-accent">&#9679;</div>
    <div class="ss-content">
      <div class="ss-title">Checkpoint Management</div>
      <div class="ss-desc">Every stream requires its own checkpoint location. You must configure and maintain these paths for each layer of the pipeline.</div>
    </div>
    <div class="ss-tag">Complexity</div>
  </div>

  <div class="ss-row">
    <div class="ss-accent">&#9679;</div>
    <div class="ss-content">
      <div class="ss-title">foreachBatch + MERGE for Upserts</div>
      <div class="ss-desc">Simple overwrites become multi-step logic: define a function, create a temp view, write a MERGE statement, and wire it into <code style="background: #EEEDE9; padding: 2px 6px; border-radius: 4px;">foreachBatch</code>.</div>
    </div>
    <div class="ss-tag">Complexity</div>
  </div>

  <div class="ss-row">
    <div class="ss-accent">&#9679;</div>
    <div class="ss-content">
      <div class="ss-title">Trigger and Schema Configuration</div>
      <div class="ss-desc">You must choose trigger modes, define schema locations for Auto Loader, and manage stream lifecycle (start, stop, await termination).</div>
    </div>
    <div class="ss-tag">Complexity</div>
  </div>

  <div class="ss-row">
    <div class="ss-accent">&#9679;</div>
    <div class="ss-content">
      <div class="ss-title">Still No Built-In Quality or Observability</div>
      <div class="ss-desc">Structured Streaming solves incremental processing, but you still have no data quality constraints, no pipeline UI, and no lineage view.</div>
    </div>
    <div class="ss-tag">Gap</div>
  </div>

  <div class="ss-takeaway">
    <div class="ss-takeaway-text">
      <strong>The takeaway:</strong> Structured Streaming solves the "re-read everything" problem, but it doesn't solve data quality, observability, or operational simplicity. You get incremental processing at the cost of writing and maintaining significantly more plumbing code. Spark Declarative Pipelines give you incremental processing <em>and</em> everything else, declaratively.
    </div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>
<summary>Click to expand</summary>

<h3>What Structured Streaming Actually Requires</h3>

To convert the batch pipeline above into an incremental streaming pipeline, here's what the bronze and silver layers would look like:

```python
# Checkpoint locations for every stream
checkpoint_base = "/Volumes/.../checkpoints"

# Bronze: readStream + writeStream + trigger + checkpoint
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/bronze_schema")
    .load(source_path)
    .writeStream
    .option("checkpointLocation", f"{checkpoint_base}/bronze")
    .trigger(availableNow=True)
    .toTable("bronze_orders")
)

# Silver: readStream from bronze + foreachBatch for merge logic
def upsert_to_silver(batch_df, batch_id):
    batch_df.createOrReplaceTempView("updates")
    batch_df.sparkSession.sql("""
        MERGE INTO silver_orders AS target
        USING updates AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

(spark.readStream
    .table("bronze_orders")
    .writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", f"{checkpoint_base}/silver")
    .trigger(availableNow=True)
    .start()
)
```

<h3>Checkpoint Management</h3>
<ul>
<li>Every streaming query needs a unique, persistent checkpoint location.</li>
<li>Checkpoints track which data has been processed so far, so the stream can resume where it left off.</li>
<li>If the checkpoint gets deleted or corrupted, the stream either reprocesses everything or fails.</li>
<li>In a multi-layer pipeline (bronze, silver, gold), you manage a separate checkpoint path for each layer.</li>
</ul>

<h3>foreachBatch + MERGE for Upserts</h3>
<ul>
<li>Structured Streaming's default output modes (append, complete, update) don't support MERGE/upsert logic directly.</li>
<li>To do incremental upserts, you need <code>foreachBatch</code>: a callback function that receives each micro-batch as a DataFrame.</li>
<li>Inside that function, you register a temp view and run a SQL MERGE statement manually.</li>
<li>This is boilerplate that every streaming silver/gold layer needs. In Spark Declarative Pipelines, you just write the query and the framework handles incremental updates.</li>
</ul>

<h3>Trigger and Schema Configuration</h3>
<ul>
<li><code>trigger(availableNow=True)</code> processes all available data then stops. Other options include <code>processingTime</code> for interval-based triggers and <code>continuous</code> for low-latency.</li>
<li>Auto Loader (<code>cloudFiles</code>) requires a <code>schemaLocation</code> so it can track and evolve the inferred schema over time.</li>
<li>You need to call <code>.start()</code> or <code>.toTable()</code> correctly, and optionally <code>.awaitTermination()</code> to block until the stream finishes.</li>
<li>Getting any of these wrong results in silent failures or streams that never terminate.</li>
</ul>

<h3>What Streaming Still Doesn't Solve</h3>
<ul>
<li><strong>Data quality:</strong> No built-in way to define expectations or constraints. You still write custom validation logic.</li>
<li><strong>Observability:</strong> No pipeline UI, no lineage graph, no per-table row counts or data quality metrics out of the box.</li>
<li><strong>Error handling:</strong> If bad data arrives, you need custom logic to quarantine or skip it. There is no "expect" or "drop" pattern.</li>
<li><strong>Operational overhead:</strong> You manage stream state, checkpoint cleanup, schema evolution, and failure recovery yourself.</li>
</ul>

<p><strong>Transition to SDP:</strong> Spark Declarative Pipelines handles all of this for you. You declare your tables, write your transformation logic, and the framework manages incremental processing, checkpoints, data quality expectations, and the pipeline UI automatically. No <code>foreachBatch</code>, no checkpoint paths, no trigger configuration.</p>

</details>


## D. Introduction to Apache Spark™ Declarative Pipelines
<br></br>
<style>
.genie-wrapper {
    display: flex;
    justify-content: center;
    align-items: center;
}
.box.genie-box {
    width: min(1100px, 100%);
    min-height: 300px;
}
.box {
    background: #F8F9FC;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(28,48,55,0.06);
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    gap: 18px;
    padding: 24px;
    text-align: center;
    position: relative;
    box-sizing: border-box;
}
.box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
    background: #00A972;
}
.genie-box::before {
    background: #FF5F46;
}
.genie-box .icon-img {
    max-width: 120px;
}
.genie-box .hero-img {
    width: 100%;
    max-width: 100%;
    height: auto;
}
.box-title {
    font-size: 26pt;
    font-weight: 700;
    color: #0b2026;
}
.three-sides {
    display: flex;
    justify-content: center;
    align-items: stretch;
    gap: 30px;
    flex-wrap: wrap;
    margin: 40px 0;
}
.small-box {
    flex: 0 0 340px;
    min-height: 260px;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    gap: 8px;
    padding: 24px 24px 20px 24px;
    text-align: left;
    position: relative;
    box-sizing: border-box;
}
.small-box::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
    background: #4299E0;
}
.box-text {
    font-size: 18pt;
    font-weight: 700;
    color: #0b2026;
    line-height: 1.3;
    text-align: center;
    margin-top: 6px;
}
.box-subtext {
    font-size: 13.5pt;
    color: #5A6F77;
    line-height: 1.45;
}
.box-subtext ul {
    margin: 0;
    padding-left: 22px;
}
.box-subtext li {
    margin-bottom: 8px;
}
.bottom-note {
    max-width: 1100px;
    margin: 0 auto 20px auto;
    padding: 18px 26px;
    background: #F8F9FC;
    border: 3px solid #FF5F46;
    border-radius: 8px;
    text-align: center;
}
.bottom-note-text {
    font-size: 1.15em;
    color: #0b2026;
    line-height: 1.5;
}
</style>

<div class="genie-wrapper">
  <div class="box genie-box">
    <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/sdp-icon.png" class="icon-img" alt="SDP Icon">
    <div class="box-title">Apache Spark™ Declarative Pipelines</div>
    <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/lakeflow-declarative-pipelines-hero.png" class="hero-img" alt="Lakeflow Hero">
  </div>
</div>

<div class="three-sides">
  <div class="small-box">
    <div class="box-text">Efficient Ingestion</div>
    <div class="box-subtext">
      <ul>
        <li><strong>Load data from any Apache Spark-supported source</strong> on Databricks</li>
        <li>Support for <strong>batch, streaming, and CDC</strong> ingestion patterns</li>
        <li>Built for <strong>data engineers, Python developers, data scientists, and SQL analysts</strong></li>
      </ul>
    </div>
  </div>
  <div class="small-box">
    <div class="box-text">Intelligent Transformation</div>
    <div class="box-subtext">
      <ul>
        <li>From just a <strong>few lines of code</strong>, pipelines are planned and executed efficiently</li>
        <li>Automatically optimizes for <strong>cost or performance</strong></li>
        <li>Reduces engineering effort by <strong>minimizing pipeline complexity</strong></li>
      </ul>
    </div>
  </div>
  <div class="small-box">
    <div class="box-text">Automated Operations</div>
    <div class="box-subtext">
      <ul>
        <li>Codifies <strong>pipeline best practices</strong> out of the box</li>
        <li>Automates <strong>dependency management, scaling, recovery, and data quality rules</strong></li>
        <li>Lets engineers focus on <strong>delivering high-quality data</strong>, not managing infrastructure</li>
      </ul>
    </div>
  </div>
</div>

<div class="bottom-note">
  <div class="bottom-note-text">
    Convert your traditional batch pipeline to <strong>Spark Declarative Pipelines (SDP)</strong> and get incremental processing, data quality enforcement, infrastructure management, and full pipeline visibility.
  </div>
</div>

[Apache Spark™ DECLARATIVE PIPELINES](https://www.databricks.com/product/data-engineering/spark-declarative-pipelines)


##### EXPAND FOR ADDITIONAL NOTES

<details>
<summary>Click to expand</summary>

<h3>Efficient Ingestion</h3>
<ul>
<li>SDP supports <strong>ingestion from any Apache Spark-supported source</strong> on Databricks.</li>
<li>Works across <strong>batch, streaming, and CDC pipelines</strong> with a unified approach.</li>
<li>Designed for multiple personas, including <strong>data engineers, Python developers, data scientists, and SQL analysts</strong>.</li>
<li>Eliminates the need to build separate ingestion frameworks for different data patterns.</li>
</ul>

<h3>Intelligent Transformation</h3>
<ul>
<li>With just a <strong>few lines of code</strong>, SDP determines how to build and execute the pipeline.</li>
<li>Automatically selects the most efficient execution strategy for <strong>batch or streaming workloads</strong>.</li>
<li>Optimizes for <strong>cost or performance</strong> without manual tuning.</li>
<li>Reduces engineering effort by <strong>abstracting pipeline complexity</strong>.</li>
</ul>

<h3>Automated Operations</h3>
<ul>
<li>SDP <strong>codifies best practices by default</strong>, reducing the need for custom engineering.</li>
<li>Automates <strong>dependency management, scaling, recovery, and data quality enforcement</strong>.</li>
<li>No infrastructure management required — pipelines are fully managed.</li>
<li>Enables teams to focus on <strong>delivering high-quality data</strong>, not operating pipelines.</li>
</ul>

</details>

### D1. Create a Pipeline Using the File Explorer

1. Complete the following steps to create a Spark Declarative Pipeline from the left navigation pane:

   a. In the left navigation bar, select the **Folder** ![Folder Icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/folder_icon.png) icon to open the Workspace navigation.

   b. Navigate to the **Build Data Pipelines with Apache Spark™ Declarative Pipelines** folder.

   c. **NOTE:** To follow along with these instructions more easily, open this notebook in a second browser tab. 
      - Right-click the notebook **02 Demo - Course Setup and Creating a Pipeline** and select **Open in new browser tab**.

   d. In the other tab, select the three-ellipsis icon ![Ellipsis Icon](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/ellipsis_icon.png) in the folder navigation bar.

   e. Select **Create** -> **ETL Pipeline**. This will bring you to the **Lakeflow Pipeline Editor**.

   f. Select **Settings** and use the following:

   | Setting | Value / Action |
   |---|---|
   | **Name** | `Test - yourfirstname-my-pipeline-project` |
   | **Default catalog** | Select your **labuser** catalog |
   | **Default schema** | Select your **sdp_1_bronze** schema (database) |

2. The project will open in the **Lakeflow Pipelines Editor** and look like the following:

   ![Pipeline Editor](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/creating-a-pipeline/lakeflow-editor-overview.png)

   *Overview of the Lakeflow Pipelines Editor UI:* [AWS](https://docs.databricks.com/aws/en/ldp/multi-file-editor#overview-of-the-lakeflow-pipelines-editor-ui) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/multi-file-editor#overview-of-the-lakeflow-pipelines-editor-ui) |
   [GCP](https://docs.databricks.com/gcp/en/ldp/multi-file-editor#overview-of-the-lakeflow-pipelines-editor-ui)

   a. Your Spark Declarative Pipeline will open within the **Lakeflow Pipelines Editor**.

   b. By default, a folder is created with a `py` file then we need to change to `sql` file.

   c. Explore the **Lakeflow Pipelines Editor** and notice the following:
      - The Spark Declarative Pipeline is located within the **Pipeline** tab.

      - To navigate back to all your files and folders, select **All Files**.

      - Write your code in the file editor.

      - View the **Pipeline graph** panel by selecting the ![Graph](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/pipeline-graph-icon.png) icon.

      - The bottom window displays pipeline run information, data previews, and performance details.

   d. Close the tab.

   **NOTE:** You will explore the pipeline editor in depth and run a pipeline in the next demonstration.

### D2. Create a Pipeline Using the Pipeline UI

1. You can also create a Spark Declarative Pipeline from the main left navigation bar in **Jobs & Pipelines**. 

   a. On the far-left navigation bar, right-click **Jobs and Pipelines** and select **Open Link in New Tab**.

   b. Select the blue **Create** button.

   c. Here you can select **ETL pipeline**. **No need to create another pipeline for this demonstration.**

## E. Conclusion

In this demonstration, you:

1. Initialized the course environment and confirmed your catalog, schemas, and source volume.
2. Previewed raw JSON order data using the `read_files` function.
3. Built a traditional ETL pipeline across bronze, silver, and gold layers and identified its key limitations at scale.
4. Explored how Apache Spark™ Declarative Pipelines address those limitations through efficient ingestion, intelligent transformation, and automated operations.
5. Created a Spark Declarative Pipeline using both the File Explorer and the Pipeline UI.

### Next Steps

In the next demonstration, you will open the Lakeflow Pipelines Editor, write your first declarative pipeline code, and run the pipeline to see incremental processing and the pipeline graph in action.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
